### Text Generation with LSTM
---

#### 1. Why Sequential Models?
The problem with BoW / TF-IDF / Feedforward NNs:

They treat text as an unordered bag. For classification it works, but for generation it collapses.

Consider: "the cat sat on the"  → next word?

- Bag of words sees: {the, cat, sat, on} - no order.
- But order matters! "sat on the" strongly predicts "mat", "floor", "chair".

We need a model that remembers what came before. That's a Recurrent Neural Network (RNN).

---

#### 2. The Recurrent Neural Network (RNN)

An **RNN** processes a sequence step-by-step. It maintains an internal state that acts as a "memory," allowing information from previous steps to influence the current output. 

At each time step $t$, the network performs the following calculation:

$$h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$$

Breakdown of Variables:
* **$x_t$**: The input at time $t$ (e.g., a character vector or word embedding).
* **$h_{t-1}$**: The previous hidden state essentially "everything the network has seen so far."
* **$h_t$**: The updated hidden state after processing the current input.
* **$W_x, W_h, b$**: These are the weights and bias. Crucially, these parameters are **shared** across all time steps, which is what defines the "recurrent" nature of the architecture.

---

Visually Unrolled:



```text
  x₁       x₂       x₃       x₄
  ↓        ↓        ↓        ↓
[ h₁ ] → [ h₂ ] → [ h₃ ] → [ h₄ ] → 


#### 3. The Vanishing Gradient Problem
To train an RNN, we use Backpropagation Through Time (BPTT) - Gradients flow backward from the loss, through every time step, to update the weights

At each backward step the gradient gets multiplied by W_h (and the derivative of tanh)

- If [W_h] < 1 -> gardient shrinks exponentially -> vanishes
- If [W_h] > 1 -> gradient grows exponentially -> explodes

Consequences: vanilla RNNs cannot learn dependencies that spam more than ~ 10 time steps. They forget the beginning of a sentence by the time they reach the end

---

#### 4. LSTM - The Solution

LSTM (Long Short-Term Memory, Horchreiter, & Schmidhuber, 1997) introduces a cell state C_t - a "memory highway" that carries information across many times steps with minimal interference. 

Three gates control what flows in and out of the cell state. Each gate is a sigmoid 

1. Forget gate (f_t) -> Decides whether to erase from the previous cell state 
2. Input gate (i_t) -> Decides what new information to add
3. Output (o_t) -> Decides what to read out as the new hidden state 

The math (intuition):

1. f_t = σ(W_f · [h_{t-1}, x_t] + b_f)         # forget valve
2. i_t = σ(W_i · [h_{t-1}, x_t] + b_i)         # input valve
3. C̃_t = tanh(W_c · [h_{t-1}, x_t] + b_c)      # candidate new content
4. C_t = f_t * C_{t-1} + i_t * C̃_t             # update cell state
5. o_t = σ(W_o · [h_{t-1}, x_t] + b_o)         # output valve
6. h_t = o_t * tanh(C_t)                       # new hidden state

- Why this fixes vanishing gradients:

The cell state C_t is updated additively (f_t * C_{t-1} + i_t * C̃_t), not multiplicatively. Gradients can flow through C_t almost unchanged across many time steps, so the model can learn long-range dependencies (50, 100+ steps).

---

#### 5. Text Generation as Next-Token Prediction
We frame generation as a classification problem:
- Given the previous N characters, predict the next character 

After traninig, we generate text by:
1. Start with a seed string
2. Predict the next character
3. Append it to the seed, drop the oldest character 
4. Repeat 

---

#### 6. Practical Implementation

##### Step 1: Imports

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
# LSTM recurrent layer that implements the gates, keras hides the math 
# Dense is a standard fully-connected layer 
from tensorflow.keras.optimizers import Adam # Adam is an adaptive optimizer that adjusts learning rates


### Code Explanation (Step-by-Step)

- `import numpy as np` → imports required library.
- `import tensorflow as tf` → imports required library.
- `from tensorflow.keras.models import Sequential` → executes step.
- `from tensorflow.keras.layers import LSTM, Dense` → executes step.
- `# LSTM recurrent layer that implements the gates, keras hides the math` → executes step.
- `# Dense is a standard fully-connected layer` → executes step.
- `from tensorflow.keras.optimizers import Adam # Adam is an adaptive optimizer that adjusts learning rates` → executes step.


##### Step 2: Load Shakespeare Dataset


In [5]:
path = tf.keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')

with open(path, 'r', encoding = 'utf-8') as f:
    text = f.read()

text = text.lower()
text = text[:100000]

print("Total characters in corpus:", len(text))
print("First 300 characters:\n", text[:300])

Total characters in corpus: 100000
First 300 characters:
 first citizen:
before we proceed any further, hear me speak.

all:
speak, speak.

first citizen:
you are all resolved rather to die than to famish?

all:
resolved. resolved.

first citizen:
first, you know caius marcius is chief enemy to the people.

all:
we know't, we know't.

first citizen:
let us


### Code Explanation (Step-by-Step)

- `path = tf.keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')` → assigns value.
- `with open(path, 'r', encoding = 'utf-8') as f:` → assigns value.
- `text = f.read()` → assigns value.
- `text = text.lower()` → assigns value.
- `text = text[:100000]` → assigns value.
- `print("Total characters in corpus:", len(text))` → prints output.
- `print("First 300 characters:\n", text[:300])` → prints output.


##### Step 3 - Build the Character Vocabulary

In [7]:
chars = sorted(list(set(text)))

char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

vocab_size = len(chars)
print("Unique characters:", vocab_size)
print("Vocabulary:", chars)

Unique characters: 37
Vocabulary: ['\n', ' ', '!', '&', "'", ',', '-', '.', ':', ';', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


### Code Explanation (Step-by-Step)

- `chars = sorted(list(set(text)))` → assigns value.
- `char_to_idx = {c: i for i, c in enumerate(chars)}` → assigns value.
- `idx_to_char = {i: c for i, c in enumerate(chars)}` → assigns value.
- `vocab_size = len(chars)` → assigns value.
- `print("Unique characters:", vocab_size)` → prints output.
- `print("Vocabulary:", chars)` → prints output.


##### Step 4: Create Input/Target Sequences 

In [10]:
seq_length = 40 # The model will read characters of context before predicting the 31st
step = 3 # We slide window forward by 1 character at a time 

sequences = []
next_chars =[]

for i in range(0, len(text)- seq_length, step):
    sequences.append(text[i : i + seq_length])
    next_chars.append(text[i + seq_length])

print("Number of training examples:", len(sequences))
print("Example input :", repr(sequences[0]))
print("Example target:", repr(next_chars[0]))

Number of training examples: 33320
Example input : 'first citizen:\nbefore we proceed any fur'
Example target: 't'


### Code Explanation (Step-by-Step)

- `seq_length = 40 # The model will read characters of context before predicting the 31st` → assigns value.
- `step = 3 # We slide window forward by 1 character at a time` → assigns value.
- `sequences = []` → assigns value.
- `next_chars =[]` → assigns value.
- `for i in range(0, len(text)- seq_length, step):` → loop to iterate.
- `sequences.append(text[i : i + seq_length])` → executes step.
- `next_chars.append(text[i + seq_length])` → executes step.
- `print("Number of training examples:", len(sequences))` → prints output.
- `print("Example input :", repr(sequences[0]))` → prints output.
- `print("Example target:", repr(next_chars[0]))` → prints output.


##### Step 5: One-Hot Encode the Sequences


In [12]:
# Create an empty 3D array filled with zeros
# Axis 0 =  number of training samples
# Axis 1 = time steps(30)
# Axis 2: vocabulary size (one-hot vector dimension)
# This 3-D shape (samples, timesteps, features) is exactly what Keras LSTM layers expect.
X = np.zeros((len(sequences), seq_length, vocab_size), dtype=np.float32) 

#2-D array for targets. Each row will be a one-hot vector marking the correct next character.
y = np.zeros((len(sequences), vocab_size), dtype=np.float32)

for i, seq in enumerate(sequences):
    for t, char in enumerate(seq):
        X[i,t, char_to_idx[char]] = 1 # set a single 1 at the position corresponding to this character. Everything else stays 0. This is one-hot encoding.
    y[i, char_to_idx[next_chars[i]]] = 1 # mark the correct target character with a 1.

print("X shape:", X.shape)
print("y shape", y.shape)

X shape: (33320, 40, 37)
y shape (33320, 37)


### Code Explanation (Step-by-Step)

- `# Create an empty 3D array filled with zeros` → executes step.
- `# Axis 0 =  number of training samples` → assigns value.
- `# Axis 1 = time steps(30)` → assigns value.
- `# Axis 2: vocabulary size (one-hot vector dimension)` → executes step.
- `# This 3-D shape (samples, timesteps, features) is exactly what Keras LSTM layers expect.` → executes step.
- `X = np.zeros((len(sequences), seq_length, vocab_size), dtype=np.float32)` → assigns value.
- `#2-D array for targets. Each row will be a one-hot vector marking the correct next character.` → executes step.
- `y = np.zeros((len(sequences), vocab_size), dtype=np.float32)` → assigns value.
- `for i, seq in enumerate(sequences):` → loop to iterate.
- `for t, char in enumerate(seq):` → loop to iterate.
- `X[i,t, char_to_idx[char]] = 1 # set a single 1 at the position corresponding to this character. Everything else stays 0. This is one-hot encoding.` → assigns value.
- `y[i, char_to_idx[next_chars[i]]] = 1 # mark the correct target character with a 1.` → assigns value.
- `print("X shape:", X.shape)` → prints output.
- `print("y shape", y.shape)` → prints output.


##### Step 6: Build the LSTM Model

In [ ]:
# 128 -> number of hidden units
model = Sequential([
    LSTM(128, input_shape = (seq_length, vocab_size)), # tells Keras the shape of one input example: 30 time steps, each a one-hot vector of size vocab_size
    Dense(vocab_size, activation='softmax') # Softmax turns the raw outputs into a probability distribution that sums to 1.
])

model.compile(
    loss = 'categorical_crossentropy',
    optimizer = Adam(learning_rate=0.01)
)

model.summary()

C:\Users\14jay\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │        84,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 37)             │         4,773 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 89,765 (350.64 KB)

 Trainable params: 89,765 (350.64 KB)

 Non-trainable params: 0 (0.00 B)

### Code Explanation (Step-by-Step)

- `# 128 -> number of hidden units` → executes step.
- `model = Sequential([` → assigns value.
- `LSTM(128, input_shape = (seq_length, vocab_size)), # tells Keras the shape of one input example: 30 time steps, each a one-hot vector of size vocab_size` → assigns value.
- `Dense(vocab_size, activation='softmax') # Softmax turns the raw outputs into a probability distribution that sums to 1.` → assigns value.
- `])` → executes step.
- `model.compile(` → executes step.
- `loss = 'categorical_crossentropy',` → assigns value.
- `optimizer = Adam(learning_rate=0.01)` → assigns value.
- `)` → executes step.
- `model.summary()` → executes step.


##### Step 7: Train the Model

In [15]:
history = model.fit(X, y, batch_size=128, epochs=20, verbose=1)

Epoch 1/20
261/261 ━━━━━━━━━━━━━━━━━━━━ 12s 38ms/step - loss: 2.4417
Epoch 2/20
261/261 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - loss: 1.9187
Epoch 3/20
261/261 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - loss: 1.7365
Epoch 4/20
261/261 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - loss: 1.6214
Epoch 5/20
261/261 ━━━━━━━━━━━━━━━━━━━━ 11s 42ms/step - loss: 1.5371
Epoch 6/20
261/261 ━━━━━━━━━━━━━━━━━━━━ 11s 40ms/step - loss: 1.4617
Epoch 7/20
261/261 ━━━━━━━━━━━━━━━━━━━━ 11s 41ms/step - loss: 1.3926
Epoch 8/20
261/261 ━━━━━━━━━━━━━━━━━━━━ 11s 41ms/step - loss: 1.3402
Epoch 9/20
261/261 ━━━━━━━━━━━━━━━━━━━━ 10s 40ms/step - loss: 1.2863
Epoch 10/20
261/261 ━━━━━━━━━━━━━━━━━━━━ 11s 42ms/step - loss: 1.2440
Epoch 11/20
261/261 ━━━━━━━━━━━━━━━━━━━━ 11s 42ms/step - loss: 1.2100
Epoch 12/20
261/261 ━━━━━━━━━━━━━━━━━━━━ 12s 44ms/step - loss: 1.1749
Epoch 13/20
261/261 ━━━━━━━━━━━━━━━━━━━━ 11s 42ms/step - loss: 1.1528
Epoch 14/20
261/261 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - loss: 1.1227
Epoch 15/20
261/261 ━━━━━━━━━

### Code Explanation (Step-by-Step)

- `history = model.fit(X, y, batch_size=128, epochs=20, verbose=1)` → assigns value.


##### Step 8: Sampling Function

In [16]:
# takes the model's probability output and a temperature value, returns the chosen character index.
def sample(preds, temprature = 1.0):
    # cast to higher precision
    preds = np.asarray(preds).astype('float64')
    # take the logarithm. Adding 1e-8 (a tiny number) prevents log(0) = -∞ errors
    preds = np.log(preds + 1e-8)/ temprature
    #  undo the log to get back to (rescaled) probabilities.
    exp_preds = np.exp(preds)
    # renormalize so they sum to 1 (now it's a valid probability distribution again).
    preds = exp_preds/ np.sum(exp_preds)
    # randomly pick an index, with each index having probability preds[i]. This is stochastic sampling
    return np.random.choice(len(preds), p = preds)

### Code Explanation (Step-by-Step)

- `# takes the model's probability output and a temperature value, returns the chosen character index.` → returns result.
- `def sample(preds, temprature = 1.0):` → defines a function.
- `# cast to higher precision` → executes step.
- `preds = np.asarray(preds).astype('float64')` → assigns value.
- `# take the logarithm. Adding 1e-8 (a tiny number) prevents log(0) = -∞ errors` → assigns value.
- `preds = np.log(preds + 1e-8)/ temprature` → assigns value.
- `#  undo the log to get back to (rescaled) probabilities.` → executes step.
- `exp_preds = np.exp(preds)` → assigns value.
- `# renormalize so they sum to 1 (now it's a valid probability distribution again).` → executes step.
- `preds = exp_preds/ np.sum(exp_preds)` → assigns value.
- `# randomly pick an index, with each index having probability preds[i]. This is stochastic sampling` → executes step.
- `return np.random.choice(len(preds), p = preds)` → returns result.


##### Step 9: Generation Function


In [24]:
def generate_text(seed, length=200, temprature = 0.5):
    generated = seed
    for _ in range(length):
        x_pred = np.zeros((1, seq_length, vocab_size))
        for t, char in enumerate(seed):
            if char in char_to_idx:
                x_pred[0, t, char_to_idx[char]] = 1

        preds = model.predict(x_pred, verbose=0)[0] 
        next_idx = sample(preds, temprature)
        next_char = idx_to_char[next_idx]

        generated += next_char
        seed = seed[1:] + next_char
    return generated

### Code Explanation (Step-by-Step)

- `def generate_text(seed, length=200, temprature = 0.5):` → defines a function.
- `generated = seed` → assigns value.
- `for _ in range(length):` → loop to iterate.
- `x_pred = np.zeros((1, seq_length, vocab_size))` → assigns value.
- `for t, char in enumerate(seed):` → loop to iterate.
- `if char in char_to_idx:` → checks condition.
- `x_pred[0, t, char_to_idx[char]] = 1` → assigns value.
- `preds = model.predict(x_pred, verbose=0)[0]` → assigns value.
- `next_idx = sample(preds, temprature)` → assigns value.
- `next_char = idx_to_char[next_idx]` → assigns value.
- `generated += next_char` → assigns value.
- `seed = seed[1:] + next_char` → assigns value.
- `return generated` → returns result.


##### Step 10: Generate Text

In [25]:
seed = text[:seq_length]
print("SEED:", repr(seed))
print()

for temp in [0.2, 0.5, 1.0]:
    print(f"--- Temperature {temp} ---")
    print(generate_text(seed, length=300, temprature=temp))
    print()

SEED: 'first citizen:\nbefore we proceed any fur'

--- Temperature 0.2 ---
first citizen:
before we proceed any furtience, there's some for the marks
and if i had made the greater for the matter wounded-!

brutus:
i have have so dispribunes, where he shall be so,
i have had have some worthy and your pats
him bether the people,
when what you shall have so dispribunes,
when worthy gone! where he shall had have som

--- Temperature 0.5 ---
first citizen:
before we proceed any furtime pat of his caunterant,
and you shall this bluck that beser the matter were son
me't of have have some i sown with bution your prove of his: they hath the for the mire to the common us i place for the people,
when wis i can and the gabore to such a servath,
when son which we we have heart, the c

--- Temperature 1.0 ---
first citizen:
before we proceed any furtius the what's in ster's thee?

marcius:
we beturnes, who,' have not save which side
not has ofdings
fonpyy witus lesscionce!

volumnia:
speak the peopl

### Code Explanation (Step-by-Step)

- `seed = text[:seq_length]` → assigns value.
- `print("SEED:", repr(seed))` → prints output.
- `print()` → prints output.
- `for temp in [0.2, 0.5, 1.0]:` → loop to iterate.
- `print(f"--- Temperature {temp} ---")` → prints output.
- `print(generate_text(seed, length=300, temprature=temp))` → prints output.
- `print()` → prints output.


#### 7: Conclusion 

- RNNs read sequences step by step, maintaining a hidden state as memory - but they suffer from vanishing gradients that prevent learning long-range patterns.
- LSTM solves this with a gated cell state: forget, input, and output gates control information - flow. The additive cell-state update lets gradients flow back through many time steps unchanged.
- Text generation is framed as repeated next-character prediction: encode the seed → predict probabilities → sample → slide the window → repeat.
- Temperature sampling controls the trade-off between safety and creativity. Low temperature is repetitive; high temperature is creative but error-prone.
- Limitations: results depend heavily on corpus size. Our small text demonstrates the mechanism; real applications use millions of characters and stack multiple LSTM layers (or use Transformers, which solved the long-range problem differently).